In [1]:
from sqlitesearch import TextSearchIndex  # персистентный поисковый индекс на базе SQLite

sqlite_index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],  # поля для полнотекстового поиска
    keyword_fields=['course'],                      # поле для точной фильтрации по курсу
    db_path='faq.db'                               # путь к файлу БД, созданному persistent_rag_ingest.ipynb
)

In [2]:
sqlite_index.count()

167

In [3]:
results = sqlite_index.search('Can I still join the course after it started?', num_results=5)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard

In [4]:
results = sqlite_index.search('Can I still join the course after it started?', num_results=5)
print([doc['question'] for doc in results])

['I just discovered the course. Can I still join?', 'I just discovered the course. Can I still join?', 'I missed the first homework - can I still get a certificate?', 'I missed the first homework - can I still get a certificate?', 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?']


In [5]:
from dotenv import load_dotenv  # для загрузки OPENAI_API_KEY из .env
load_dotenv()                   # добавляет переменные из .env в os.environ

from openai import OpenAI       # клиент OpenAI API
openai_client = OpenAI()        # автоматически берёт OPENAI_API_KEY из окружения

In [6]:
from rag_helper import RAGBase  # импортируем класс RAG-пайплайна

assistant = RAGBase(sqlite_index, openai_client)  # передаём sqlite_index вместо minsearch (интерфейс одинаковый)

In [7]:
assistant.rag('I just discovered the course, can I still join?')  # запрос к персистентному индексу из faq.db

'Yes, you can still join. If you want a certificate, you need to submit your project while submissions are still open.'

In [8]:
sqlite_index.close()

In [9]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. **Install Ollama**\n   - **macOS:** download and install the `.pkg` from [ollama.com/download](https://ollama.com/download)\n   - **Windows:** download and install the `.msi`\n   - **Linux:** run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the LLaMA 3 model, starts it locally, and opens a chat interface.\n\n3. **Test the local server**\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": your_prompt}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the Ollama server with:\n\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```'

In [10]:
assistant.rag("How do I run Ollma locally?")

'I don’t know.'